### This notebook will focus on finding the best model to predict the severity predictor !

In [3]:
### Libraries we will be using
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, accuracy_score,recall_score,f1_score,roc_curve,roc_auc_score, precision_recall_curve,confusion_matrix,ConfusionMatrixDisplay,classification_report, mean_absolute_error, cohen_kappa_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb


In [4]:
## Data set we will be using 
data = pd.read_csv('../Datasets/pre-processedData.csv')
attack_data = data[data['is_malicious'] == 1]
attack_data.drop(columns=['is_malicious', 'Unnamed: 0'], inplace = True)
attack_data.head(5)

/var/folders/bt/9_3kc80d4jq8sv8fv_c7d_fr0000gn/T/ipykernel_39868/3148040238.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attack_data.drop(columns=['is_malicious', 'Unnamed: 0'], inplace = True)


,duration,src_bytes,dst_bytes,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,is_guest_login,count,srv_count,serror_rate,rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,level,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,Web_services,File_services,remote_login_service,email_service,dns_service,icmp_service,netbios_service,database_service,diagnostic_service,auth_service,messaging_service,other_service,flag_REJ,flag_RSTO,flag_RSTR,flag_S1,flag_S3,flag_SF,flag_SH,Severity_Score,bytes_ratio,total_bytes
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,4.820282,1.945910,1.0,0.0,0.05,0.07,0.0,255,26,0.10,0.05,0.0,0.0,1.0,0.0,0.0,19,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,4.804021,2.995732,0.0,1.0,0.16,0.06,0.0,255,19,0.07,0.07,0.0,0.0,0.0,1.0,1.0,21,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,2,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,5.117994,2.302585,1.0,0.0,0.05,0.06,0.0,255,9,0.04,0.05,0.0,0.0,1.0,0.0,0.0,21,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,4.770685,2.833213,1.0,0.0,0.14,0.06,0.0,255,15,0.06,0.07,0.0,0.0,1.0,0.0,0.0,21,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,5.602119,3.178054,1.0,0.0,0.09,0.05,0.0,255,23,0.09,0.05,0.0,0.0,1.0,0.0,0.0,21,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0.0,0.0


In [5]:
## Train test split of the data 
RANDOM_SEED = 42
X = attack_data.drop(columns=['Severity_Score'])
y = attack_data['Severity_Score']

## Splitting the data into train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=RANDOM_SEED)


---

### Model 1 : Logisitc Regression 

In [56]:
### Setting custom class weights based on the severity type 
class_weights = {1:1,2:3, 3:10} 
pipeline_steps = [('scaler', StandardScaler()),('logit', LogisticRegression(solver='lbfgs', class_weight = class_weights,C=1000,penalty='l2',max_iter = 1000))]
logit_pipeline = Pipeline(pipeline_steps)

logit_model = logit_pipeline.fit(X_train,y_train)

/opt/anaconda3/envs/Network_Intrusion/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [57]:
logit_training_pred = logit_model.predict(X_train)
logit_testing_pred = logit_model.predict(X_test)
logit_training_proba = logit_model.predict_proba(X_train)[:,1]
logit_testing_proba = logit_model.predict_proba(X_test)[:,1]


In [93]:
## Evaluating the model
logit_mae = mean_absolute_error(y_test, logit_testing_pred)
print(f"{logit_mae:.3f}")

# Cohen Kappa Score 
cks = cohen_kappa_score(y_test, logit_testing_pred)
print(f"{cks:.2f}")


high_severe_testing_data = y_test == 3
high_severe_predicted_test_data = logit_testing_pred == 3

### Confusion Matrix for the severe cases 

true_positives = ((y_test == 3)& (logit_testing_pred == 3)).sum()
false_positives = ((y_test != 3) & (logit_testing_pred == 3)).sum()
false_negatives = ((y_test == 3) & (logit_testing_pred != 3)).sum()
true_negatives = ((y_test != 3) & (logit_testing_pred != 3)).sum()



### Metrics for calculation 
logit_rare_precision = true_positives/(true_positives+false_positives) 
logit_rare_recall = true_positives/(true_positives + false_negatives)
logit_rare_fpr = false_positives/(true_negatives+false_positives) ## Misclassified
logit_rare_fnr = false_negatives/(true_positives+false_negatives) ## False Alaram

print(logit_rare_precision, logit_rare_recall, logit_rare_fpr, logit_rare_fnr)




0.007
0.99
0.9756986634264885 0.9901356350184957 0.001483459427384661 0.009864364981504316


---
### Model 2: Linear SVM 



In [28]:
pipeline_steps = [('scaler', StandardScaler()), ('svm', LinearSVC(multi_class='ovr', penalty='l2', C = 0.1))]
svm_pipeline = Pipeline(pipeline_steps)

best_svm = svm_pipeline.fit(X_train, y_train)

In [ ]:
svm_testing_pred = best_svm.predict(X_test)
svm_training_pred = best_svm.predict(X_train)

svm_testing_boundary = best_svm.decision_function(X_test)
svm_training_boundary = best_svm.decision_function(X_train)

In [45]:
### Metrics for evaluation 
svm_mae = mean_absolute_error(y_test, svm_testing_pred)

## Cohen Kappa Score 
svm_cohen_kappa = cohen_kappa_score(y_test, svm_testing_pred)

## Confusion Matrix 

true_positives = ((y_test == 3) & (svm_testing_pred == 3)).sum()
true_negatives = ((y_test != 3) & (svm_testing_pred != 3)).sum()
false_negatives = ((y_test == 3) & (svm_testing_pred != 3)).sum()
false_positives = ((y_test != 3) & (svm_testing_pred == 3 )).sum()


### Precision
svm_precision_rare = true_positives /(true_positives + false_positives)

## Recall
svm_recall_rare = true_positives/(true_positives + false_negatives)

## F1_score 

svm_f1_rare = 2*((svm_precision_rare * svm_recall_rare)/(svm_precision_rare + svm_recall_rare))

### False Positive Rate

svm_fpr = false_positives / (false_positives + true_negatives) ## Misclassified
svm_fnr = false_negatives / (false_negatives + true_positives) ## False Alaram


print(svm_precision_rare, svm_recall_rare, svm_f1_rare, svm_fpr, svm_fnr)

0.9887359198998749 0.9741060419235512 0.9813664596273292 0.0006675567423230974 0.025893958076448828


---
## Naive Bayes 

In [61]:
naive_model = GaussianNB(var_smoothing=1e-9)
naive_model.fit(X_train, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [91]:
## Training and testing pred

naive_testing_pred = naive_model.predict(X_test)
naive_training_pred = naive_model.predict(X_train)

naive_testing_proba = naive_model.predict_proba(X_test)
naive_training_proba = naive_model.predict_proba(X_train)


In [73]:
## Metrics for evaluation 

naive_mae = mean_absolute_error(y_test, naive_testing_pred)

## Cohen-Kappa Score

naive_cohen = cohen_kappa_score(y_test, naive_testing_pred)

## Confusion Matrix 

true_positives = ((y_test == 3) & (naive_testing_pred == 3)).sum()
true_negatives = ((y_test != 3) & (naive_testing_pred != 3)).sum()
false_positives = ((y_test != 3) & (naive_testing_pred == 3)).sum()
false_negatives = ((y_test == 3) & (naive_testing_pred != 3)).sum()

### Precision, Recall, F1-Score, fpr, fnr 

naive_precision = true_positives / (true_positives + false_positives)
naive_recall = true_positives / (true_positives + false_negatives)

naive_fpr = false_positives /(false_positives + true_negatives) ## Misclassified Cases
naive_fnr = false_negatives /(false_negatives + true_positives) ## False Alarm 



print(naive_mae, naive_cohen, naive_precision, naive_recall, naive_fpr, naive_fnr)

0.07535157069894353 0.8457851351117285 0.8656174334140436 0.8816276202219482 0.008233199821984869 0.11837237977805179


---

### Model 4: Decision Tree Classifier 

In [89]:
decision_model = DecisionTreeClassifier(criterion='gini', random_state=RANDOM_SEED, max_depth = 10, max_features=12, min_samples_leaf=75, min_samples_split=230)

decision_model.fit(X_train,y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",230
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",75
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",12
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current 

In [ ]:
decision_testing_pred = decision_model.predict(X_test)
decision_training_pred = decision_model.predict(X_train)

decision_testing_proba = decision_model.predict_proba(X_test)[:,1]
decision_training_proba = decision_model.predict_proba(X_train)[:,1]

[0. 1. 1. ... 1. 1. 0.]


In [99]:
### Metrics for the evaluation 

## Mean Absoulute Error 
decision_mae = mean_absolute_error(y_test, decision_testing_pred)

## Cohen Kappa's 


decision_cohen = cohen_kappa_score(y_test, decision_testing_pred)

## Confusion matrix 

true_positives = ((y_test ==3) & (decision_testing_pred == 3)).sum()
true_negatives = ((y_test != 3) & (decision_testing_pred != 3)).sum()
false_positives = ((y_test != 3) & (decision_testing_pred == 3)).sum()
false_negatives = ((y_test == 3) & (decision_testing_pred != 3)).sum()


## Precision, recall and f1 score 


decision_precision = true_positives / (true_positives + false_positives)
decision_recall = true_positives / (true_positives + false_negatives)
decision_fpr = false_positives  / (false_positives + true_negatives) ##Mis classification error 
decision_fnr = false_negatives / (false_negatives + true_positives) ## False Alarm
decision_f1 = 2 * ((decision_precision * decision_recall) / (decision_precision + decision_recall))


print(decision_precision, decision_recall, decision_fpr, decision_fnr, decision_f1)

0.9509433962264151 0.9321824907521579 0.002892745883400089 0.06781750924784218 0.9414694894146949


---
### Model 5: Random Forest Classifier

In [113]:
estimators = [100,200,300,400,500]
oob_scores = []
for n in estimators:
    random_model = RandomForestClassifier(oob_score = True , n_estimators= n, min_samples_leaf=70, min_samples_split=160, max_depth = 12, max_features=8)
    random_model.fit(X_train,y_train)
    oob_scores.append(random_model.oob_score_)

In [ ]:
random_model = RandomForestClassifier(criterion='gini', n_estimators=200, oob_score=True, max_depth = 11, max_features= 10, min_samples_leaf=70, min_samples_split=200, random_state= RANDOM_SEED)
random_model.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",11
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",200
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",70
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",10
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_